# Step 6: Hybrid Ensemble Model

In [1]:
# Import reusable model factory functions

import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models.logistic_regression import (
    create_logistic_regression_pipeline
)
from src.models.random_forest import (
    create_random_forest_pipeline
)
from src.models.xgboost import create_xgboost_pipeline
from src.models.ensemble import (
    create_weighted_soft_voting_ensemble
)

print("Model modules imported successfully.")

Model modules imported successfully.


In [2]:
# Create all model objects using the reusable .py modules

logistic_pipeline = create_logistic_regression_pipeline()
random_forest_pipeline = create_random_forest_pipeline()
xgboost_pipeline = create_xgboost_pipeline()

hybrid_ensemble = create_weighted_soft_voting_ensemble(
    logistic_model=logistic_pipeline,
    random_forest_model=random_forest_pipeline,
    xgboost_model=xgboost_pipeline,
    weights=(1, 2, 2)
)

print("All model objects created from .py files.")

All model objects created from .py files.


In [3]:
# Load the previously created train, validation, and test splits

import pandas as pd

split_folder = project_root / "data" / "processed" / "splits"

X_train = pd.read_csv(split_folder / "X_train.csv")
X_validation = pd.read_csv(split_folder / "X_validation.csv")
X_test = pd.read_csv(split_folder / "X_test.csv")

y_train = pd.read_csv(
    split_folder / "y_train.csv"
).squeeze("columns")

y_validation = pd.read_csv(
    split_folder / "y_validation.csv"
).squeeze("columns")

y_test = pd.read_csv(
    split_folder / "y_test.csv"
).squeeze("columns")

print("Data splits loaded.")
print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

Data splits loaded.
Training: (151858, 548)
Validation: (50620, 548)
Test: (50620, 548)


In [4]:
# Train the baseline models created from the .py files

logistic_pipeline.fit(X_train, y_train)
random_forest_pipeline.fit(X_train, y_train)
xgboost_pipeline.fit(X_train, y_train)

print("All baseline models trained.")

All baseline models trained.


In [5]:
# Train the weighted soft-voting ensemble

hybrid_ensemble.fit(X_train, y_train)

print("Hybrid ensemble trained successfully.")

Hybrid ensemble trained successfully.


In [6]:
# Evaluate the hybrid ensemble on validation data

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

hybrid_validation_predictions = hybrid_ensemble.predict(X_validation)

hybrid_validation_metrics = {
    "model": "Weighted Soft-Voting Ensemble",
    "split": "validation",
    "accuracy": accuracy_score(
        y_validation,
        hybrid_validation_predictions
    ),
    "precision": precision_score(
        y_validation,
        hybrid_validation_predictions
    ),
    "recall": recall_score(
        y_validation,
        hybrid_validation_predictions
    ),
    "f1_score": f1_score(
        y_validation,
        hybrid_validation_predictions
    )
}

print(hybrid_validation_metrics)

tn, fp, fn, tp = confusion_matrix(
    y_validation,
    hybrid_validation_predictions
).ravel()

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

{'model': 'Weighted Soft-Voting Ensemble', 'split': 'validation', 'accuracy': 0.9673251679178191, 'precision': 0.981677796327212, 'recall': 0.9508813065976714, 'f1_score': 0.9660341711844915}

TN: 25445
FP: 439
FN: 1215
TP: 23521


In [7]:
# Save the trained hybrid ensemble for later evaluation and prediction

import joblib

models_folder = project_root / "models"
models_folder.mkdir(parents=True, exist_ok=True)

hybrid_model_path = (
    models_folder / "weighted_soft_voting_ensemble.joblib"
)

joblib.dump(hybrid_ensemble, hybrid_model_path)

print("Hybrid ensemble saved to:")
print(hybrid_model_path)

Hybrid ensemble saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\models\weighted_soft_voting_ensemble.joblib
